# 01 – Mortgage Dataset Validation

## Dissertation: Intelligent Data Quality Monitoring

### Objective
To assess the structural integrity, completeness, consistency
and validity of the fabricated mortgage reporting dataset.

### Data sources
- Mortgage account base
- Monthly aggregated mortgage metrics
- Data quality execution log
- Ground-truth anomaly manifest

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Display configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Display configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

Matplotlib is building the font cache; this may take a moment.


Libraries imported successfully.


In [2]:
# Locate project root
PROJECT_ROOT = Path.cwd().parent

# Dataset location
DATA_DIR = PROJECT_ROOT / "data" / "output"

# Load CSV files
accounts = pd.read_csv(
    DATA_DIR / "mortgage_accounts.csv",
    parse_dates=["open_date"]
)

monthly_metrics = pd.read_csv(
    DATA_DIR / "monthly_metrics.csv"
)

dq_log = pd.read_csv(
    DATA_DIR / "dq_run_log.csv"
)

anomalies = pd.read_csv(
    DATA_DIR / "anomalies_manifest.csv"
)

print("All four datasets loaded successfully.")

All four datasets loaded successfully.


In [3]:
datasets = {
    "Mortgage Accounts": accounts,
    "Monthly Metrics": monthly_metrics,
    "DQ Run Log": dq_log,
    "Anomaly Manifest": anomalies
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 40)
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]:,}")


Mortgage Accounts
----------------------------------------
Rows: 20,000
Columns: 9

Monthly Metrics
----------------------------------------
Rows: 36
Columns: 15

DQ Run Log
----------------------------------------
Rows: 2,772
Columns: 9

Anomaly Manifest
----------------------------------------
Rows: 13
Columns: 6


In [4]:
for name, df in datasets.items():

    print(f"\n{name.upper()}")
    print(df.dtypes)


MORTGAGE ACCOUNTS
account_id                    str
open_date          datetime64[us]
region                        str
product_subtype               str
balance                   float64
ltv_band                      str
arrears_status                str
channel                       str
_ltv_numeric              float64
dtype: object

MONTHLY METRICS
month                       str
new_mortgages_count       int64
balance_sum             float64
avg_ltv                 float64
active_rate             float64
arrears_rate            float64
product_fixed_prop      float64
product_tracker_prop    float64
product_svr_prop        float64
region_capital_prop     float64
region_zealand_prop     float64
region_southern_prop    float64
region_central_prop     float64
region_northern_prop    float64
region_western_prop     float64
dtype: object

DQ RUN LOG
run_id              str
month               str
metric_name         str
rule_type           str
status              str
rows_checked      

In [5]:
for name, df in datasets.items():

    print(f"\n{name.upper()}")

    missing = df.isnull().sum()

    missing = missing[missing > 0]

    if len(missing) > 0:
        print(missing)
    else:
        print("No missing values detected.")


MORTGAGE ACCOUNTS
No missing values detected.

MONTHLY METRICS
avg_ltv    1
dtype: int64

DQ RUN LOG
threshold_used    1440
dtype: int64

ANOMALY MANIFEST
No missing values detected.


In [6]:
print("ACCOUNT DUPLICATES")
print(accounts["account_id"].duplicated().sum())

print("\nMONTHLY METRIC DUPLICATES")
print(monthly_metrics["month"].duplicated().sum())

print("\nDQ RUN ID DUPLICATES")
print(dq_log["run_id"].duplicated().sum())

ACCOUNT DUPLICATES
0

MONTHLY METRIC DUPLICATES
0

DQ RUN ID DUPLICATES
0


In [7]:
monthly_metrics["month"] = pd.to_datetime(
    monthly_metrics["month"]
)

monthly_metrics = monthly_metrics.sort_values("month")

expected_months = pd.date_range(
    start=monthly_metrics["month"].min(),
    periods=36,
    freq="MS"
)

actual_months = pd.DatetimeIndex(
    monthly_metrics["month"]
)

print("Expected monthly observations:", len(expected_months))
print("Actual monthly observations:", len(actual_months))

print(
    "Missing reporting months:",
    len(expected_months.difference(actual_months))
)

print(
    "Unexpected reporting months:",
    len(actual_months.difference(expected_months))
)

Expected monthly observations: 36
Actual monthly observations: 36
Missing reporting months: 0
Unexpected reporting months: 0


In [8]:
product_columns = [
    "product_fixed_prop",
    "product_tracker_prop",
    "product_svr_prop"
]

region_columns = [
    col for col in monthly_metrics.columns
    if col.startswith("region_")
    and col.endswith("_prop")
]

print("PRODUCT PROPORTION CHECK")
print(
    monthly_metrics[product_columns].sum(axis=1).describe()
)

print("\nREGION PROPORTION CHECK")
print(
    monthly_metrics[region_columns].sum(axis=1).describe()
)

PRODUCT PROPORTION CHECK
count   36.0000
mean     1.0000
std      0.0000
min      1.0000
25%      1.0000
50%      1.0000
75%      1.0000
max      1.0000
dtype: float64

REGION PROPORTION CHECK
count   36.0000
mean     1.0000
std      0.0000
min      1.0000
25%      1.0000
50%      1.0000
75%      1.0000
max      1.0000
dtype: float64


In [9]:
print("GROUND-TRUTH ANOMALIES")

display(
    anomalies.sort_values(
        ["anomaly_type", "month"]
    )
)

GROUND-TRUTH ANOMALIES


,month,metric_name,anomaly_type,description,should_fixed_rule_catch,should_learned_method_catch
8,2025-03-01,product_subtype_proportions,CATEGORY_DRIFT,Gradual migration from Fixed towards Tracker a...,False,True
9,2025-04-01,product_subtype_proportions,CATEGORY_DRIFT,Gradual migration from Fixed towards Tracker a...,False,True
10,2025-05-01,product_subtype_proportions,CATEGORY_DRIFT,Gradual migration from Fixed towards Tracker a...,False,True
11,2025-06-01,product_subtype_proportions,CATEGORY_DRIFT,Gradual migration from Fixed towards Tracker a...,False,True
0,2024-10-01,active_rate,LEVEL_SHIFT,Sudden downward step change of 3.5 percentage ...,True,True
12,2025-09-01,avg_ltv,MISSING_COLUMN,avg_ltv unavailable for this reporting month.,True,True
7,2025-04-01,new_mortgages_count,SEASONAL_FALSE_ALARM,Legitimate spring origination peak exceeding t...,True,False
1,2024-04-01,arrears_rate,SLOW_DRIFT,Gradual cumulative arrears increase; individua...,False,True
2,2024-05-01,arrears_rate,SLOW_DRIFT,Gradual cumulative arrears increase; individua...,False,True
3,2024-06-01,arrears_rate,SLOW_DRIFT,Gradual cumulative arrears increase; individua...,False,True
